# Web Scraper 

## Project Goals
### 1. Scrapes book data from books.toscrape.com using BeautifulSoup and Python. Stores results in CSV for analysis

## Tools Used
### 1. Python
### 2. Requests
### 3. BeautifulSoup
### 4. CSV module
### 5. HTML inspection

In [23]:
import requests ## HTTP requests
from bs4 import BeautifulSoup  ## Webscraping library
import csv  ## Store data into csv file 
import datetime  ## Create data objects 


In [24]:
def generate_url(page_number=1):
    if page_number == 1:
        return "https://books.toscrape.com/"
    else:
        return f"https://books.toscrape.com/catalogue/page-{page_number}.html"


In [75]:
def extract_book_data(book_card):  ## Takes in a BeautifulSoup Tag object of class bs4.element.Tag
                                ## With the tag, can retrieve the text, nested emements, tag attributes, and print HTML string
                                ## <div> Groups related elements, <p> paragraph of text, <h[number]> A heading/title 
            try:
                title = book_card.h3.a['title']
            except:
                title = ""
            try:
                price = book_card.find("p", class_ = "price_color").text.strip()
            except = price = ""
            
            try:
                availability = book_card.("p", class_ = "instock availability").text.strip()
            except:
                availability = ""
            
            return {
        "title": title,
        "price": price,
        "availability": availability}
            
    

In [108]:
# Step 1: Get the page
url = "https://books.toscrape.com/"
headers = {
    "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) "
                  "AppleWebKit/537.36 (KHTML, like Gecko) "
                  "Chrome/122.0.0.0 Safari/537.36"
}

response = requests.get(url, headers=headers)

soup = BeautifulSoup(response.text, "html.parser")

# Step 3: Find the first book card
first_book = soup.find("article", class_="product_pod")

In [109]:
# Step 2: Extract book data from single card

def extract_book_data(book_card):
    """
    Takes in a BeautifulSoup Tag object of class bs4.element.Tag
    With the tag, can retrieve:
      - the text content
      - nested elements (e.g., a tag inside an h3)
      - tag attributes (like 'href' or 'title')
      - and even print the HTML string
    
    HTML tags in use:
      - <div>: Groups related elements
      - <p>: Paragraph of text
      - <h3>: A heading/title
    """
    try:
        title = book_card.h3.a['title']
    except:
        title = ""
    
    try:
        price = book_card.find("p", class_="price_color").text.strip()
        price = price.replace("Â£", "£")

    except:
        price = ""
    
    try:
        availability = book_card.find("p", class_="instock availability").text.strip()
    except:
        availability = ""
    
    return {
        "title": title,
        "price": price,
        "availability": availability
    }


In [117]:
# Step 3: Look through all data on a page and save to CSV
def main(pages_to_scrape=1):
    base_url = "https://books.toscrape.com/"
    all_books = []  ## Empty list but soon to be list of dictionaries 
    
    for page in range(1, pages_to_scrape + 1):
        url = generate_url(page)
        response = requests.get(url)
        response.encoding = 'utf-8'
        
        if response.status_code == 200:
            soup = BeautifulSoup(response.text, "html.parser") ## Converts raw HTML into BeautifulSoup object to search and extract
            book_cards = soup.find_all("article", class_ = "product_pod")
            
            for card in book_cards: 
                book_data = extract_book_data(card)
                all_books.append(book_data)
        else:
            print(f"Failed to load page {page}: Status {response.status_code}")
        
    filename = f"books_{datetime.datetime.now().strftime('%Y-%m-%d')}.csv"
    with open(filename, "w", newline="", encoding="utf-8-sig") as f:
        writer = csv.DictWriter(f, fieldnames=["title", "price", "availability"])
        writer.writeheader()
        writer.writerows(all_books)

    print(f" Saved {len(all_books)} books to {filename}")

In [118]:
main(pages_to_scrape=2)  # Scrapes the first 2 pages of books


 Saved 40 books to books_2025-04-14.csv
